# Step 1: Extract dopaminergic neurons from GSE178265

This notebook downloads the large human single-nucleus file from GEO (Kamath et al. 2022), keeps only the dopaminergic (DA) neuron candidates plus a random sample of 20,000 other cells, and saves a small file you can analyse on your Mac.

**How to run:** click each grey code box from top to bottom and press **Shift + Enter**. Wait for each one to finish (the spinning circle stops and a green tick appears) before running the next.

**Total time:** about 1–2 hours. Keep this browser tab open the whole time, or Colab may disconnect.

## Cell 1 — Download the three files from GEO (10–20 minutes)
You will see progress bars. The big file is 4.7 GB.

In [ ]:
!mkdir -p /content/data
%cd /content/data
base = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE178nnn/GSE178265/suppl"
for name in ["GSE178265_Homo_features.tsv.gz", "GSE178265_Homo_bcd.tsv.gz", "GSE178265_Homo_matrix.mtx.gz"]:
    !wget -c -q --show-progress {base}/{name}
!ls -lh /content/data

## Cell 2 — Load the helper functions (a few seconds)
Nothing to check here; it just prepares the tools.

In [ ]:
# Core logic for extracting dopaminergic-neuron candidates from a large
# Matrix Market (.mtx.gz) file without loading the whole matrix into memory.
# The Colab notebook runs these same steps cell by cell.

import gzip
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.io import mmwrite

CHUNK = 20_000_000  # rows of the .mtx file read at a time


def read_header(mtx_path):
    """Return (n_rows, n_cols, n_entries) and the number of lines before the data."""
    skip = 0
    with gzip.open(mtx_path, "rt") as f:
        for line in f:
            skip += 1
            if line.startswith("%"):
                continue
            n_rows, n_cols, n_entries = (int(x) for x in line.split())
            return n_rows, n_cols, n_entries, skip
    raise ValueError("No size line found in the .mtx file")


def iter_chunks(mtx_path, skip):
    """Yield (row, col, value) arrays, 0-based, one chunk at a time."""
    reader = pd.read_csv(
        mtx_path, sep=" ", header=None, skiprows=skip, chunksize=CHUNK,
        names=["r", "c", "v"], dtype={"r": np.int64, "c": np.int64, "v": np.float32},
        compression="gzip", engine="c",
    )
    for chunk in reader:
        yield (chunk["r"].to_numpy() - 1, chunk["c"].to_numpy() - 1, chunk["v"].to_numpy())


def load_names(features_path, barcodes_path):
    feats = pd.read_csv(features_path, sep="\t", header=None, dtype=str)
    bcds = pd.read_csv(barcodes_path, sep="\t", header=None, dtype=str)
    return feats, bcds


def find_gene_column(feats, markers):
    """Pick the features column that contains the marker gene symbols."""
    best, best_hits = 0, -1
    for j in range(feats.shape[1]):
        hits = feats[j].isin(markers).sum()
        if hits > best_hits:
            best, best_hits = j, hits
    return best, best_hits


def orientation(n_rows, n_cols, n_feats, n_bcds):
    """'genes_by_cells' or 'cells_by_genes'."""
    if n_rows == n_feats and n_cols == n_bcds:
        return "genes_by_cells"
    if n_rows == n_bcds and n_cols == n_feats:
        return "cells_by_genes"
    raise ValueError(f"Matrix is {n_rows} x {n_cols}, but there are {n_feats} features and {n_bcds} barcodes")


def pass1_marker_counts(mtx_path, skip, orient, n_cells, marker_idx):
    """Per-cell total counts and per-cell counts for each marker gene."""
    totals = np.zeros(n_cells, dtype=np.float64)
    marker_counts = {g: np.zeros(n_cells, dtype=np.float64) for g in marker_idx}
    idx_to_gene = {i: g for g, i in marker_idx.items()}
    marker_set = np.array(list(idx_to_gene.keys()))
    for r, c, v in iter_chunks(mtx_path, skip):
        genes, cells = (r, c) if orient == "genes_by_cells" else (c, r)
        totals += np.bincount(cells, weights=v, minlength=n_cells)
        hit = np.isin(genes, marker_set)
        for gi, ci, vi in zip(genes[hit], cells[hit], v[hit]):
            marker_counts[idx_to_gene[gi]][ci] += vi
    return totals, marker_counts


def choose_cells(totals, marker_counts, min_umi=500, n_reference=20_000, seed=1):
    """DA candidates = any DA marker detected; reference = random sample of the rest."""
    da_signal = sum(marker_counts.values())
    ok = totals >= min_umi
    da = np.where(ok & (da_signal > 0))[0]
    rest = np.where(ok & (da_signal == 0))[0]
    rng = np.random.default_rng(seed)
    ref = rng.choice(rest, size=min(n_reference, rest.size), replace=False)
    return np.sort(da), np.sort(ref)


def pass2_extract(mtx_path, skip, orient, n_genes, n_cells, keep):
    """Keep only entries for the selected cells; return a genes x kept-cells matrix."""
    new_pos = np.full(n_cells, -1, dtype=np.int64)
    new_pos[keep] = np.arange(keep.size)
    rows, cols, vals = [], [], []
    for r, c, v in iter_chunks(mtx_path, skip):
        genes, cells = (r, c) if orient == "genes_by_cells" else (c, r)
        m = new_pos[cells] >= 0
        rows.append(genes[m].astype(np.int32))
        cols.append(new_pos[cells[m]].astype(np.int32))
        vals.append(v[m])
    rows, cols, vals = np.concatenate(rows), np.concatenate(cols), np.concatenate(vals)
    return sparse.csc_matrix((vals, (rows, cols)), shape=(n_genes, keep.size))


def write_outputs(out_dir, mat, feats, gene_col, barcodes, cell_info):
    import os
    os.makedirs(out_dir, exist_ok=True)
    with gzip.open(f"{out_dir}/matrix.mtx.gz", "wb") as f:
        mmwrite(f, mat.tocoo())
    feats[[gene_col]].to_csv(f"{out_dir}/features.tsv.gz", sep="\t", header=False, index=False)
    pd.Series(barcodes).to_csv(f"{out_dir}/barcodes.tsv.gz", sep="\t", header=False, index=False)
    cell_info.to_csv(f"{out_dir}/cell_info.csv", index=False)


## Cell 3 — Inspect the files (1–2 minutes)
**Check the printout:** the four marker genes should be found (`found 4 of 4`), and the layout line should say `genes_by_cells` or `cells_by_genes`. If you see an error instead, copy it and send it to Claude.

In [ ]:
MTX = "/content/data/GSE178265_Homo_matrix.mtx.gz"
FEATS = "/content/data/GSE178265_Homo_features.tsv.gz"
BCDS = "/content/data/GSE178265_Homo_bcd.tsv.gz"
MARKERS = ["TH", "SLC6A3", "SLC18A2", "DDC"]

n_rows, n_cols, n_entries, skip = read_header(MTX)
feats, bcds = load_names(FEATS, BCDS)
gene_col, hits = find_gene_column(feats, MARKERS)
orient = orientation(n_rows, n_cols, len(feats), len(bcds))
marker_idx = {g: int(feats.index[feats[gene_col] == g][0]) for g in MARKERS if (feats[gene_col] == g).any()}
barcodes = bcds[0].to_numpy()

print(f"Matrix: {n_rows:,} x {n_cols:,} with {n_entries:,} non-zero values")
print(f"Genes: {len(feats):,}   Cells: {len(bcds):,}")
print(f"Layout: {orient}")
print(f"Marker genes found {hits} of {len(MARKERS)}: {list(marker_idx)}")
print("Example barcodes:", list(barcodes[:3]))

## Cell 4 — First pass: count marker genes in every cell (20–40 minutes)
This reads the whole file once, in pieces, without loading it all into memory.

In [ ]:
import time
t0 = time.time()
totals, marker_counts = pass1_marker_counts(MTX, skip, orient, len(barcodes), marker_idx)
print(f"Done in {(time.time() - t0) / 60:.1f} minutes")
for g, v in marker_counts.items():
    print(f"{g}: detected in {(v > 0).sum():,} cells")

## Cell 5 — Choose the cells to keep (seconds)
**Check the printout:** you should see roughly 20,000–60,000 DA candidates. If the number is above 80,000, the cell automatically switches to a stricter rule (TH plus a second DA marker) so the next step fits in memory.

In [ ]:
da, ref = choose_cells(totals, marker_counts, min_umi=500, n_reference=20_000)
if da.size > 80_000:
    strict = (marker_counts["TH"] > 0) & ((marker_counts.get("SLC6A3", 0) > 0) | (marker_counts.get("SLC18A2", 0) > 0)) & (totals >= 500)
    da = np.where(strict)[0]
    print("Loose rule gave too many cells; switched to the strict rule (TH plus SLC6A3 or SLC18A2).")
keep = np.concatenate([da, ref])
print(f"DA candidates: {da.size:,}")
print(f"Reference sample of other cells: {ref.size:,}")
print(f"Total cells kept: {keep.size:,} of {len(barcodes):,}")

## Cell 6 — Second pass: pull out the kept cells and save them (30–60 minutes)

In [ ]:
t0 = time.time()
sub = pass2_extract(MTX, skip, orient, len(feats), len(barcodes), keep)
cell_info = pd.DataFrame({
    "barcode": barcodes[keep],
    "group": ["DA_candidate"] * da.size + ["reference_sample"] * ref.size,
    "total_umi": totals[keep],
    **{f"{g}_counts": marker_counts[g][keep] for g in marker_counts},
})
OUT = "/content/GSE178265_DA_extract"
write_outputs(OUT, sub, feats, gene_col, barcodes[keep], cell_info)
print(f"Done in {(time.time() - t0) / 60:.1f} minutes")
!ls -lh {OUT}

## Cell 7 — Save to your Google Drive (5–10 minutes)
A pop-up will ask you to allow Colab to access your Google Drive. Click **Connect to Google Drive** and approve. The result lands in **My Drive → GSE178265_DA_extract.zip**, which you then download to your Mac.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cd /content && zip -r -q GSE178265_DA_extract.zip GSE178265_DA_extract
!cp /content/GSE178265_DA_extract.zip /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive/GSE178265_DA_extract.zip
print("Finished. Download GSE178265_DA_extract.zip from your Google Drive to your Mac.")